In [1]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 0 │ Mount Google Drive · Load YOLOv11 Weights         ║
# ╚══════════════════════════════════════════════════════════════╝

!pip install ultralytics -q

import os
from pathlib import Path
from google.colab import drive
from ultralytics import YOLO

# ── Mount Drive ───────────────────────────────────────────────
drive.mount("/content/drive")

# ── Path to your trained weights ─────────────────────────────
DRIVE_WEIGHTS = Path(
    "/content/drive/MyDrive/PPE_YOLO11/yolov11n_results/"
    "yolov11n_v1/weights/best.pt"
)

if not DRIVE_WEIGHTS.exists():
    raise FileNotFoundError(
        f"Weights not found at:\n{DRIVE_WEIGHTS}\n"
        "Check your Drive path and update DRIVE_WEIGHTS."
    )

print(f"✔  Weights found: {DRIVE_WEIGHTS}")

# ── Load model ────────────────────────────────────────────────
print("Loading model into memory…")
model = YOLO(str(DRIVE_WEIGHTS))

print("\n── Model Ready ──")
print(f"   Task        : {model.task}")
print(f"   Classes     : {model.names}")
print(f"\n✔  CELL 0 COMPLETE — model variable is live.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✔  Weights found: /content/drive/MyDrive/PPE_YOLO11/yolov11n_results/yolov11n_v1/weights/best.pt
Loading model into memory…

── Model Ready ──
   Task        : detect
   Classes     : {0: 'Hardhat', 1: 'Mask', 2: 'NO-Hardhat', 3: 'NO-Mask', 4: 'NO-Safety Vest', 5: 'Person', 6: 'Safety Cone', 7: 'Safety Vest', 8: 'machinery', 9: 'vehicle'}

✔  CELL 0 COMPLETE — model variable is live.


In [2]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 1 │ Imports · Design Tokens · CSS · PDF Engine        ║
# ╚══════════════════════════════════════════════════════════════╝

!pip install reportlab -q

import cv2
import numpy as np
from PIL import Image
import gradio as gr
import io, os, time, random, datetime
from dataclasses import dataclass, field
from typing import Optional

from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.units import mm
from reportlab.platypus import (
    SimpleDocTemplate, Table, TableStyle,
    Paragraph, Spacer, HRFlowable, KeepTogether
)
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_CENTER
from reportlab.pdfgen import canvas as rl_canvas

print("✔  Imports loaded — Gradio", gr.__version__)

# ── Design tokens ─────────────────────────────────────────────
PALETTE = {
    "bg_main":       "#0F172A",
    "bg_panel":      "#1E293B",
    "bg_panel_alt":  "#162032",
    "border":        "#334155",
    "border_accent": "#475569",
    "text_primary":  "#F1F5F9",
    "text_secondary":"#94A3B8",
    "text_mono":     "#CBD5E1",
    "accent_blue":   "#3B82F6",
    "accent_cyan":   "#06B6D4",
    "compliant":     "#10B981",
    "semi":          "#F59E0B",
    "violation":     "#EF4444",
    "compliant_dim": "#064E3B",
    "semi_dim":      "#78350F",
    "violation_dim": "#7F1D1D",
    "header_bar":    "#0B1120",
    "grid_row_alt":  "#162032",
}
FONT_MONO = "'JetBrains Mono','Fira Code','Courier New',monospace"
FONT_SANS = "'Inter','Segoe UI',system-ui,sans-serif"

print("✔  Design tokens registered.")

# ── Industrial CSS ────────────────────────────────────────────
INDUSTRIAL_CSS = f"""
@import url('https://fonts.googleapis.com/css2?family=JetBrains+Mono:wght@400;500;700&family=Inter:wght@300;400;500;600;700&display=swap');

*,*::before,*::after{{box-sizing:border-box;margin:0;padding:0}}

body,.gradio-container{{
    background-color:{PALETTE['bg_main']}!important;
    font-family:{FONT_SANS}!important;
    color:{PALETTE['text_primary']}!important;
    min-height:100vh;
}}
.gradio-container>.main>.wrap{{background:transparent!important;padding:0!important}}

.ppe-header{{
    background:{PALETTE['header_bar']};
    border-bottom:2px solid {PALETTE['accent_blue']};
    padding:14px 24px;
    display:flex;align-items:center;justify-content:space-between;
    position:sticky;top:0;z-index:100;
}}
.ppe-header-title{{
    font-family:{FONT_MONO};font-size:15px;font-weight:700;
    color:{PALETTE['accent_cyan']};letter-spacing:0.08em;text-transform:uppercase;
}}
.ppe-header-subtitle{{
    font-family:{FONT_SANS};font-size:11px;
    color:{PALETTE['text_secondary']};margin-top:2px;
}}
.sys-status-online{{
    display:inline-flex;align-items:center;gap:6px;
    background:{PALETTE['compliant_dim']};border:1px solid {PALETTE['compliant']};
    color:{PALETTE['compliant']};font-family:{FONT_MONO};font-size:10px;
    font-weight:700;letter-spacing:0.1em;padding:4px 10px;
    border-radius:2px;text-transform:uppercase;
}}
.sys-status-online::before{{
    content:'';width:7px;height:7px;border-radius:50%;
    background:{PALETTE['compliant']};box-shadow:0 0 6px {PALETTE['compliant']};
    animation:pulse-green 2s infinite;
}}
@keyframes pulse-green{{0%,100%{{opacity:1}}50%{{opacity:0.4}}}}

.metric-grid{{
    display:grid;grid-template-columns:repeat(4,1fr);
    gap:10px;margin-bottom:12px;
}}
.metric-card{{
    background:{PALETTE['bg_panel']};border:1px solid {PALETTE['border']};
    border-top:2px solid {PALETTE['accent_blue']};border-radius:0;padding:14px 16px;
}}
.metric-card-value{{
    font-family:{FONT_MONO};font-size:28px;font-weight:700;
    color:{PALETTE['text_primary']};line-height:1;margin-bottom:4px;
}}
.metric-card-label{{
    font-family:{FONT_SANS};font-size:10px;font-weight:500;
    color:{PALETTE['text_secondary']};letter-spacing:0.08em;text-transform:uppercase;
}}
.metric-card-sub{{
    font-family:{FONT_MONO};font-size:10px;
    color:{PALETTE['accent_cyan']};margin-top:6px;
}}
.metric-card.compliant {{border-top-color:{PALETTE['compliant']};}}
.metric-card.semi      {{border-top-color:{PALETTE['semi']};}}
.metric-card.violation {{border-top-color:{PALETTE['violation']};}}
.metric-card.compliant .metric-card-value{{color:{PALETTE['compliant']};}}
.metric-card.semi      .metric-card-value{{color:{PALETTE['semi']};}}
.metric-card.violation .metric-card-value{{color:{PALETTE['violation']};}}

.badge{{
    display:inline-block;font-family:{FONT_MONO};font-size:10px;font-weight:700;
    letter-spacing:0.1em;padding:3px 8px;border-radius:2px;
    text-transform:uppercase;white-space:nowrap;
}}
.badge-compliant{{background:{PALETTE['compliant_dim']};color:{PALETTE['compliant']};border:1px solid {PALETTE['compliant']};}}
.badge-semi     {{background:{PALETTE['semi_dim']};color:{PALETTE['semi']};border:1px solid {PALETTE['semi']};}}
.badge-violation{{background:{PALETTE['violation_dim']};color:{PALETTE['violation']};border:1px solid {PALETTE['violation']};}}

.telem-grid{{width:100%;border-collapse:collapse;font-family:{FONT_MONO};font-size:12px;}}
.telem-grid thead tr{{background:{PALETTE['bg_main']};border-bottom:2px solid {PALETTE['accent_blue']};}}
.telem-grid thead th{{color:{PALETTE['accent_cyan']};font-size:10px;font-weight:700;letter-spacing:.12em;text-transform:uppercase;padding:8px 12px;text-align:left;}}
.telem-grid tbody tr{{border-bottom:1px solid {PALETTE['border']};}}
.telem-grid tbody tr:nth-child(odd) {{background:{PALETTE['bg_panel']};}}
.telem-grid tbody tr:nth-child(even){{background:{PALETTE['grid_row_alt']};}}
.telem-grid tbody tr:hover          {{background:{PALETTE['border']};}}
.telem-grid tbody td{{padding:8px 12px;color:{PALETTE['text_mono']};vertical-align:middle;}}
.telem-grid tbody td.id-col{{color:{PALETTE['accent_blue']};font-weight:700;}}

.telem-strip{{
    background:{PALETTE['bg_main']};
    border-top:1px solid {PALETTE['border']};
    border-bottom:1px solid {PALETTE['border']};
    padding:6px 16px;font-family:{FONT_MONO};font-size:10px;
    color:{PALETTE['text_secondary']};letter-spacing:.06em;
    display:flex;gap:24px;flex-wrap:wrap;
}}
.telem-strip span.live{{color:{PALETTE['accent_cyan']};}}

.ppe-panel-title{{
    font-family:{FONT_MONO};font-size:11px;font-weight:700;
    color:{PALETTE['accent_cyan']};letter-spacing:.12em;text-transform:uppercase;
    border-bottom:1px solid {PALETTE['border']};
    padding-bottom:8px;margin-bottom:12px;
}}

.tab-nav{{background:{PALETTE['header_bar']}!important;border-bottom:1px solid {PALETTE['border']}!important;border-radius:0!important;gap:0!important;}}
.tab-nav button{{background:transparent!important;border:none!important;border-right:1px solid {PALETTE['border']}!important;border-bottom:3px solid transparent!important;border-radius:0!important;color:{PALETTE['text_secondary']}!important;font-family:{FONT_MONO}!important;font-size:11px!important;font-weight:600!important;letter-spacing:.08em!important;text-transform:uppercase!important;padding:12px 20px!important;transition:all 0.15s!important;}}
.tab-nav button:hover   {{color:{PALETTE['text_primary']}!important;background:{PALETTE['bg_panel']}!important;}}
.tab-nav button.selected{{color:{PALETTE['accent_cyan']}!important;border-bottom-color:{PALETTE['accent_cyan']}!important;background:{PALETTE['bg_panel']}!important;}}

.gradio-container .block{{background:transparent!important;border:none!important;box-shadow:none!important;}}
.gradio-container table{{background:{PALETTE['bg_panel']}!important;border-radius:0!important;}}
.gradio-container table th{{background:{PALETTE['bg_main']}!important;color:{PALETTE['accent_cyan']}!important;font-family:{FONT_MONO}!important;font-size:10px!important;letter-spacing:.1em!important;text-transform:uppercase!important;border-bottom:2px solid {PALETTE['accent_blue']}!important;}}
.gradio-container table td{{font-family:{FONT_MONO}!important;font-size:12px!important;color:{PALETTE['text_mono']}!important;border-bottom:1px solid {PALETTE['border']}!important;}}

::-webkit-scrollbar{{width:6px;height:6px;}}
::-webkit-scrollbar-track{{background:{PALETTE['bg_main']};}}
::-webkit-scrollbar-thumb{{background:{PALETTE['border_accent']};border-radius:2px;}}
"""
print("✔  CSS compiled.")

# ── PDF colour palette (white bg, print-safe) ─────────────────
_C = {
    "black":    colors.HexColor("#0F172A"),
    "heading":  colors.HexColor("#1E3A5F"),
    "subtext":  colors.HexColor("#475569"),
    "rule":     colors.HexColor("#CBD5E1"),
    "comply":   colors.HexColor("#059669"),
    "semi":     colors.HexColor("#B45309"),
    "violate":  colors.HexColor("#B91C1C"),
    "accent":   colors.HexColor("#1D4ED8"),
    "row_alt":  colors.HexColor("#F8FAFC"),
    "row_head": colors.HexColor("#1E3A5F"),
    "badge_c":  colors.HexColor("#D1FAE5"),
    "badge_s":  colors.HexColor("#FEF3C7"),
    "badge_v":  colors.HexColor("#FEE2E2"),
    "border_t": colors.HexColor("#E2E8F0"),
    "white":    colors.white,
    "ltblue":   colors.HexColor("#93C5FD"),
}

def _pdf_header(canv, doc):
    W, H = A4
    canv.setFillColor(_C["heading"])
    canv.rect(0, H-28*mm, W, 28*mm, fill=1, stroke=0)
    canv.setStrokeColor(colors.HexColor("#3B82F6"))
    canv.setLineWidth(1.5)
    canv.line(0, H-28*mm, W, H-28*mm)
    canv.setFont("Helvetica-Bold", 13)
    canv.setFillColor(_C["white"])
    canv.drawString(15*mm, H-12*mm, "PPE COMPLIANCE MONITOR")
    canv.setFont("Helvetica", 8)
    canv.setFillColor(_C["ltblue"])
    canv.drawString(15*mm, H-20*mm, "Industrial Safety Intelligence System")
    ts = datetime.datetime.now().strftime("%Y-%m-%d  %H:%M:%S  UTC+3")
    canv.setFillColor(_C["white"])
    canv.drawRightString(W-15*mm, H-12*mm, ts)
    canv.setFillColor(_C["ltblue"])
    canv.drawRightString(W-15*mm, H-20*mm, "CONFIDENTIAL — INTERNAL USE ONLY")
    canv.setFont("Helvetica", 7)
    canv.setFillColor(_C["subtext"])
    canv.drawString(15*mm, 8*mm,
        f"PPE Compliance Monitor v2.1  ·  Doc ID: {getattr(doc,'docID','—')}")
    canv.drawRightString(W-15*mm, 8*mm, f"Page {canv.getPageNumber()}")
    canv.setStrokeColor(_C["rule"]); canv.setLineWidth(0.5)
    canv.line(15*mm, 13*mm, W-15*mm, 13*mm)

def _sc(s): return {"COMPLIANT":_C["comply"],"SEMI-COMPLIANT":_C["semi"],"NON-COMPLIANT":_C["violate"]}.get(s.upper(),_C["black"])
def _sb(s): return {"COMPLIANT":_C["badge_c"],"SEMI-COMPLIANT":_C["badge_s"],"NON-COMPLIANT":_C["badge_v"]}.get(s.upper(),_C["white"])

def generate_compliance_pdf(workers, detections,
                             compliance_score=0.0, risk_level="LOW",
                             site_name="INDUSTRIAL COMPLEX — ZONE 7",
                             inspector="AUTOMATED VISION SYSTEM"):
    buf = io.BytesIO()
    ts  = datetime.datetime.now().strftime("%Y-%m-%d  %H:%M:%S  UTC+3")
    doc = SimpleDocTemplate(buf, pagesize=A4,
          leftMargin=15*mm, rightMargin=15*mm,
          topMargin=36*mm, bottomMargin=22*mm)
    doc.docID = f"PPE-{datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}"

    total     = len(workers)
    compliant = sum(1 for w in workers if w.compliance_state=="COMPLIANT")
    semi      = sum(1 for w in workers if w.compliance_state=="SEMI-COMPLIANT")
    violation = sum(1 for w in workers if w.compliance_state=="NON-COMPLIANT")
    rate      = round(compliant/total*100,1) if total else 0.0
    W,_ = A4; usable = W-30*mm
    S = getSampleStyleSheet()
    def ps(name,**kw): return ParagraphStyle(name,parent=S["Normal"],**kw)

    story = []
    story.append(Paragraph("SAFETY COMPLIANCE INCIDENT REPORT",
        ps("T",fontName="Helvetica-Bold",fontSize=16,textColor=_C["heading"],spaceAfter=2)))
    story.append(Paragraph(f"Site: {site_name}",
        ps("M",fontName="Helvetica",fontSize=8,textColor=_C["subtext"],spaceAfter=1)))
    story.append(Paragraph(
        f"Inspector: {inspector}  ·  Timestamp: {ts}  ·  "
        f"Scene Risk: {risk_level}  ·  Compliance: {compliance_score:.1f}%",
        ps("M2",fontName="Helvetica",fontSize=8,textColor=_C["subtext"],spaceAfter=2)))
    story.append(Spacer(1,4*mm))
    story.append(HRFlowable(width="100%",thickness=1.5,color=_C["accent"],spaceAfter=6))

    # KPI
    story.append(Paragraph("SCAN SUMMARY",
        ps("SH",fontName="Helvetica-Bold",fontSize=9,textColor=_C["white"],
           backColor=_C["heading"],borderPad=(4,6,4,6),spaceBefore=8,spaceAfter=4)))
    kpi = Table(
        [["TOTAL WORKERS","COMPLIANT","SEMI-COMPLIANT","NON-COMPLIANT","COMPLIANCE RATE"],
         [str(total),str(compliant),str(semi),str(violation),f"{rate} %"]],
        colWidths=[usable/5]*5)
    kpi.setStyle(TableStyle([
        ("BACKGROUND",(0,0),(-1,0),_C["row_head"]),("TEXTCOLOR",(0,0),(-1,0),_C["white"]),
        ("FONTNAME",(0,0),(-1,0),"Helvetica-Bold"),("FONTSIZE",(0,0),(-1,0),8),
        ("ALIGN",(0,0),(-1,-1),"CENTER"),
        ("TOPPADDING",(0,0),(-1,-1),6),("BOTTOMPADDING",(0,0),(-1,-1),6),
        ("BACKGROUND",(0,1),(-1,1),_C["white"]),
        ("FONTNAME",(0,1),(-1,1),"Helvetica-Bold"),("FONTSIZE",(0,1),(-1,1),20),
        ("TOPPADDING",(0,1),(-1,1),10),("BOTTOMPADDING",(0,1),(-1,1),10),
        ("TEXTCOLOR",(1,1),(1,1),_C["comply"]),("TEXTCOLOR",(2,1),(2,1),_C["semi"]),
        ("TEXTCOLOR",(3,1),(3,1),_C["violate"]),("TEXTCOLOR",(4,1),(4,1),_C["accent"]),
        ("GRID",(0,0),(-1,-1),0.5,_C["border_t"]),("BOX",(0,0),(-1,-1),1.5,_C["heading"]),
    ]))
    story.append(kpi); story.append(Spacer(1,5*mm))

    # Worker detail
    story.append(Paragraph("WORKER DETECTION DETAIL",
        ps("SH2",fontName="Helvetica-Bold",fontSize=9,textColor=_C["white"],
           backColor=_C["heading"],borderPad=(4,6,4,6),spaceBefore=8,spaceAfter=4)))
    rows=[["WORKER","STATUS","PPE WORN","VIOLATIONS","RISK SCORE","RISK LEVEL","TIME"]]
    for w in workers:
        rows.append([
            f"W{w.worker_id}", w.compliance_state,
            ", ".join(w.ppe_worn) if w.ppe_worn else "—",
            ", ".join(w.violations) if w.violations else "—",
            f"{w.risk_score:.3f}", w.risk_level,
            datetime.datetime.now().strftime("%H:%M:%S"),
        ])
    col_w=[18*mm,38*mm,38*mm,38*mm,20*mm,18*mm,23*mm]
    det=Table(rows,colWidths=col_w,repeatRows=1)
    rstyles=[
        ("BACKGROUND",(0,0),(-1,0),_C["row_head"]),("TEXTCOLOR",(0,0),(-1,0),_C["white"]),
        ("FONTNAME",(0,0),(-1,0),"Helvetica-Bold"),("FONTSIZE",(0,0),(-1,0),7.5),
        ("ALIGN",(0,0),(-1,-1),"CENTER"),("VALIGN",(0,0),(-1,-1),"MIDDLE"),
        ("TOPPADDING",(0,0),(-1,-1),5),("BOTTOMPADDING",(0,0),(-1,-1),5),
        ("FONTNAME",(0,1),(-1,-1),"Helvetica"),("FONTSIZE",(0,1),(-1,-1),8),
        ("ROWBACKGROUNDS",(0,1),(-1,-1),[_C["white"],_C["row_alt"]]),
        ("GRID",(0,0),(-1,-1),0.5,_C["border_t"]),("BOX",(0,0),(-1,-1),1,_C["heading"]),
    ]
    for i,w in enumerate(workers,1):
        rstyles+=[("BACKGROUND",(1,i),(1,i),_sb(w.compliance_state)),
                  ("TEXTCOLOR",(1,i),(1,i),_sc(w.compliance_state)),
                  ("FONTNAME",(1,i),(1,i),"Helvetica-Bold")]
    det.setStyle(TableStyle(rstyles))
    story.append(det); story.append(Spacer(1,5*mm))

    viols=[w for w in workers if w.compliance_state!="COMPLIANT"]
    if viols:
        story.append(Paragraph("VIOLATION & SEMI-COMPLIANCE NARRATIVE",
            ps("SH3",fontName="Helvetica-Bold",fontSize=9,textColor=_C["white"],
               backColor=_C["heading"],borderPad=(4,6,4,6),spaceBefore=8,spaceAfter=4)))
        for w in viols:
            miss=", ".join(w.violations) if w.violations else "Unspecified"
            vt=Table([[
                Paragraph(f"<b>{w.compliance_state}</b>",
                    ps("VS",fontName="Helvetica-Bold",fontSize=8,textColor=_sc(w.compliance_state))),
                Paragraph(
                    f"<b>Worker W{w.worker_id}</b>  ·  Violations: "
                    f"<font color='#B91C1C'><b>{miss}</b></font>"
                    f"  ·  Risk Score: {w.risk_score:.3f}  ·  Risk Level: {w.risk_level}",
                    ps("VB",fontName="Helvetica",fontSize=8.5,textColor=_C["black"])),
            ]],colWidths=[38*mm,usable-38*mm])
            vt.setStyle(TableStyle([
                ("BACKGROUND",(0,0),(0,0),_sb(w.compliance_state)),
                ("BACKGROUND",(1,0),(1,0),_C["white"]),
                ("ALIGN",(0,0),(0,0),"CENTER"),("VALIGN",(0,0),(-1,-1),"MIDDLE"),
                ("TOPPADDING",(0,0),(-1,-1),7),("BOTTOMPADDING",(0,0),(-1,-1),7),
                ("LEFTPADDING",(0,0),(-1,-1),8),
                ("BOX",(0,0),(-1,-1),1,_C["border_t"]),
                ("LINEAFTER",(0,0),(0,-1),2,_sc(w.compliance_state)),
            ]))
            story.append(KeepTogether([vt,Spacer(1,2*mm)]))

    story.append(Spacer(1,6*mm))
    story.append(HRFlowable(width="100%",thickness=0.5,color=_C["rule"]))
    story.append(Paragraph(
        f"Auto-generated · PPE Compliance Monitor v2.1 · {doc.docID} · {ts}",
        ps("FN",fontName="Helvetica",fontSize=7,
           textColor=_C["subtext"],alignment=TA_CENTER,spaceBefore=4)))

    doc.build(story,onFirstPage=_pdf_header,onLaterPages=_pdf_header)
    result=buf.getvalue(); buf.close()
    return result

print("✔  PDF engine ready.")
print("\n══════════════════════════════════════")
print("  CELL 1 COMPLETE")
print("══════════════════════════════════════")

✔  Imports loaded — Gradio 5.50.0
✔  Design tokens registered.
✔  CSS compiled.
✔  PDF engine ready.

══════════════════════════════════════
  CELL 1 COMPLETE
══════════════════════════════════════


In [3]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 2 │ Real YOLOv11 Inference · Worker Grouping          ║
# ║  Uses the `model` variable loaded in Cell 0.                ║
# ╚══════════════════════════════════════════════════════════════╝

import cv2, numpy as np, time, datetime
from PIL import Image
from dataclasses import dataclass, field
from typing import Optional

# ── Class map — matches your trained model exactly ────────────
CLASS_NAMES = {
    0:'Hardhat', 1:'Mask', 2:'NO-Hardhat',
    3:'NO-Mask',  4:'NO-Safety Vest', 5:'Person',
    6:'Safety Cone', 7:'Safety Vest', 8:'machinery', 9:'vehicle'
}
VIOLATION_WEIGHTS = {
    'NO-Hardhat':     1.0,
    'NO-Safety Vest': 0.80,
    'NO-Mask':        0.55,
}
VIOLATION_CLASSES = set(VIOLATION_WEIGHTS.keys())
COMPLIANT_ITEMS   = {'Hardhat', 'Safety Vest', 'Mask'}
PPE_ITEMS         = VIOLATION_CLASSES | COMPLIANT_ITEMS
HAZARD_CONTEXT    = {'machinery', 'vehicle'}
SAFETY_MITIGATION = {'Safety Cone'}
RISK_THRESHOLDS   = [('CRITICAL',0.70),('HIGH',0.45),('MEDIUM',0.20),('LOW',0.00)]

COLOURS = {
    'danger':  (239, 68,  68),
    'safe':    ( 34,197,  94),
    'info':    ( 59,130, 246),
    'neutral': (148,163, 184),
    'hazard':  (251,146,  60),
    'warning': (234,179,   8),
}

# ── Worker dataclass ──────────────────────────────────────────
@dataclass
class Worker:
    worker_id:        int
    bbox:             list
    confidence:       float
    ppe_worn:         list = field(default_factory=list)
    violations:       list = field(default_factory=list)
    compliance_state: str  = 'COMPLIANT'
    risk_score:       float = 0.0
    risk_level:       str   = 'LOW'

    @property
    def status_colour(self) -> str:
        return {
            'COMPLIANT':      '#1A9E4A',
            'SEMI-COMPLIANT': '#D4A017',
            'NON-COMPLIANT':  '#D92B2B',
        }[self.compliance_state]

def _hex_bgr(h: str) -> tuple:
    h = h.lstrip('#')
    return (int(h[4:6],16), int(h[2:4],16), int(h[0:2],16))

# ── Compliance classification ─────────────────────────────────
def _classify_worker(violations: list) -> str:
    if not violations:
        return 'COMPLIANT'
    if len(violations) >= 2:
        return 'NON-COMPLIANT'
    if VIOLATION_WEIGHTS.get(violations[0], 0) >= 0.9:
        return 'NON-COMPLIANT'
    return 'SEMI-COMPLIANT'

# ── IoU / spatial helpers ─────────────────────────────────────
def _iou(a, b):
    xA=max(a[0],b[0]); yA=max(a[1],b[1])
    xB=min(a[2],b[2]); yB=min(a[3],b[3])
    inter=max(0,xB-xA)*max(0,yB-yA)
    if inter==0: return 0.0
    return inter/((a[2]-a[0])*(a[3]-a[1])+(b[2]-b[0])*(b[3]-b[1])-inter)

def _ppe_near(ppe_box, person_box, thresh=0.10):
    cx=(ppe_box[0]+ppe_box[2])/2; cy=(ppe_box[1]+ppe_box[3])/2
    inside=(person_box[0]<=cx<=person_box[2] and person_box[1]<=cy<=person_box[3])
    return inside or _iou(ppe_box, person_box) > thresh

# ── Worker grouping ───────────────────────────────────────────
def group_workers(detections: list) -> list:
    persons = [d for d in detections if d['label']=='Person']
    ppe_det = [d for d in detections if d['label'] in PPE_ITEMS]
    if not persons:
        return []
    workers = [Worker(worker_id=i+1, bbox=p['bbox'], confidence=p['confidence'])
               for i,p in enumerate(persons)]
    for ppe in ppe_det:
        scores = [(w, _iou(ppe['bbox'],w.bbox))
                  for w in workers if _ppe_near(ppe['bbox'],w.bbox)]
        if scores:
            best = max(scores, key=lambda x: x[1])[0]
        else:
            pcx=(ppe['bbox'][0]+ppe['bbox'][2])/2
            pcy=(ppe['bbox'][1]+ppe['bbox'][3])/2
            best = min(workers, key=lambda w:
                (pcx-(w.bbox[0]+w.bbox[2])/2)**2 +
                (pcy-(w.bbox[1]+w.bbox[3])/2)**2)
        if ppe['label'] in VIOLATION_CLASSES:
            best.violations.append(ppe['label'])
        else:
            best.ppe_worn.append(ppe['label'])

    for w in workers:
        w.compliance_state = _classify_worker(w.violations)
        if w.violations:
            w.risk_score = round(
                sum(w.confidence * VIOLATION_WEIGHTS[v] for v in w.violations), 4)
            w.risk_level = next(
                (lvl for lvl,th in RISK_THRESHOLDS if w.risk_score>=th), 'LOW')
        else:
            w.risk_score, w.risk_level = 0.0, 'LOW'
    return workers

# ── Scene risk ────────────────────────────────────────────────
def compute_risk(detections: list, workers: list) -> tuple:
    violations = [d for d in detections if d['label'] in VIOLATION_CLASSES]
    n_persons  = len(workers) or sum(1 for d in detections if d['label']=='Person')
    n_hazard   = sum(1 for d in detections if d['label'] in HAZARD_CONTEXT)
    n_cones    = sum(1 for d in detections if d['label'] in SAFETY_MITIGATION)
    non_c      = [w for w in workers if w.compliance_state=='NON-COMPLIANT']

    if not violations:
        bd = dict(base=0.0,machinery=0.0,density=0.0,
                  mitigation=0.0,raw=0.0,persons=n_persons,hazards=n_hazard)
        return 100.0, 0.0, 'LOW', violations, bd

    base       = sum(w.risk_score for w in workers)
    machinery  = min(n_hazard*0.10, 0.30)
    density    = max(0, len(non_c)-1)*0.05
    mitigation = min(n_cones*0.03, 0.09)
    raw        = base + machinery + density - mitigation
    risk_score = float(np.clip(raw/max(n_persons,1), 0.0, 1.0))
    compliance = round((1.0-risk_score)*100, 1)
    risk_level = next((lvl for lvl,th in RISK_THRESHOLDS if risk_score>=th), 'LOW')
    bd = dict(base=round(base,4), machinery=round(machinery,4),
              density=round(density,4), mitigation=round(mitigation,4),
              raw=round(raw,4), persons=n_persons, hazards=n_hazard)
    return compliance, round(risk_score,4), risk_level, violations, bd

# ── Frame annotation ──────────────────────────────────────────
def annotate_frame(frame: np.ndarray, detections: list,
                   workers: list, compliance: float,
                   risk_level: str) -> np.ndarray:
    """
    INPUT:  RGB numpy uint8
    OUTPUT: RGB numpy uint8  ← Gradio type='numpy' renders at native resolution
    """
    img = frame.copy()
    h, w = img.shape[:2]

    # PPE / hazard / cone boxes (non-person)
    for det in detections:
        if det['label'] == 'Person':
            continue
        x1,y1,x2,y2 = map(int, det['bbox'])
        if det['label'] in VIOLATION_CLASSES:   col = COLOURS['danger']
        elif det['label'] in COMPLIANT_ITEMS:   col = COLOURS['safe']
        elif det['label'] in HAZARD_CONTEXT:    col = COLOURS['hazard']
        else:                                   col = COLOURS['neutral']
        cv2.rectangle(img,(x1,y1),(x2,y2),col,2)
        txt = f"{det['label']} {det['confidence']:.2f}"
        (tw,th),_ = cv2.getTextSize(txt,cv2.FONT_HERSHEY_SIMPLEX,0.42,1)
        cv2.rectangle(img,(x1,y1-th-7),(x1+tw+4,y1),col,-1)
        cv2.putText(img,txt,(x1+2,y1-4),
                    cv2.FONT_HERSHEY_SIMPLEX,0.42,(255,255,255),1,cv2.LINE_AA)

    # Worker boxes with corner ticks + label below box
    for wk in workers:
        x1,y1,x2,y2 = map(int, wk.bbox)
        col  = _hex_bgr(wk.status_colour)
        tick = max(12, int((x2-x1)*0.12))
        t    = 3
        cv2.rectangle(img,(x1,y1),(x2,y2),col,2)
        for (px,py),(dx,dy) in [
            ((x1,y1),(1,0)),((x1,y1),(0,1)),
            ((x2,y1),(-1,0)),((x2,y1),(0,1)),
            ((x1,y2),(1,0)),((x1,y2),(0,-1)),
            ((x2,y2),(-1,0)),((x2,y2),(0,-1)),
        ]:
            cv2.line(img,(px,py),(px+dx*tick,py+dy*tick),col,t)
        label = f"W{wk.worker_id} {wk.compliance_state}"
        (tw,th),_ = cv2.getTextSize(label,cv2.FONT_HERSHEY_SIMPLEX,0.48,1)
        cv2.rectangle(img,(x1,y2),(x1+tw+6,y2+th+8),col,-1)
        cv2.putText(img,label,(x1+3,y2+th+3),
                    cv2.FONT_HERSHEY_SIMPLEX,0.48,(255,255,255),1,cv2.LINE_AA)

    # HUD — compliance + risk top-right
    hud_col = {'CRITICAL':COLOURS['danger'],'HIGH':(220,100,40),
               'MEDIUM':COLOURS['warning'],'LOW':COLOURS['safe']}.get(risk_level,COLOURS['neutral'])
    ok_n = sum(1 for wk in workers if wk.compliance_state=='COMPLIANT')
    for i,txt in enumerate([
        f"Compliance: {compliance:.1f}%",
        f"Risk: {risk_level}",
        f"Workers: {ok_n}/{len(workers)} OK",
    ]):
        # Black shadow for readability on any background
        cv2.putText(img,txt,(w-248,22+i*24),
                    cv2.FONT_HERSHEY_SIMPLEX,0.52,(0,0,0),3,cv2.LINE_AA)
        cv2.putText(img,txt,(w-248,22+i*24),
                    cv2.FONT_HERSHEY_SIMPLEX,0.52,hud_col,1,cv2.LINE_AA)
    return img

# ── Result container ──────────────────────────────────────────
@dataclass
class InferenceResult:
    annotated_image:      np.ndarray
    detections:           list
    workers:              list
    violations:           list
    compliance_score:     float
    risk_level:           str
    risk_score:           float
    person_count:         int
    compliant_count:      int
    semi_compliant_count: int
    non_compliant_count:  int
    summary:              str
    inference_time_ms:    float
    risk_breakdown:       dict
    timing:               dict
    frame_id:             Optional[int] = None

# ── REAL YOLO INFERENCE ───────────────────────────────────────
def run_inference(source,
                  conf_threshold: float = 0.45,
                  frame_id: Optional[int] = None) -> InferenceResult:
    """
    source: numpy RGB uint8 array (from Gradio type='numpy')
            OR PIL Image OR file path string.

    Uses `model` loaded in Cell 0 (your YOLOv11n best.pt).
    Returns InferenceResult.annotated_image as RGB numpy uint8.
    Gradio type='numpy' renders this at native resolution.
    """

    # ── Normalise to RGB numpy uint8 ─────────────────────────
    if isinstance(source, str):
        frame = cv2.cvtColor(cv2.imread(source), cv2.COLOR_BGR2RGB)
    elif isinstance(source, Image.Image):
        frame = np.array(source.convert("RGB"), dtype=np.uint8)
    elif isinstance(source, np.ndarray):
        frame = source.astype(np.uint8)
        if frame.ndim == 2:
            frame = cv2.cvtColor(frame, cv2.COLOR_GRAY2RGB)
    else:
        raise ValueError(f"Unsupported input type: {type(source)}")

    t0 = time.perf_counter()

    # ── YOUR REAL YOLO MODEL ──────────────────────────────────
    # model was loaded in Cell 0 from your Drive weights
    results = model.predict(
        source=frame,          # RGB numpy array
        conf=conf_threshold,
        verbose=False,
    )[0]

    detections = [
        {
            'label':      CLASS_NAMES.get(int(b.cls[0]), f'cls_{int(b.cls[0])}'),
            'confidence': round(float(b.conf[0]), 4),
            'bbox':       b.xyxy[0].tolist(),   # [x1,y1,x2,y2] floats
            'class_id':   int(b.cls[0]),
        }
        for b in results.boxes
    ]

    t1 = time.perf_counter()

    # ── Worker grouping + compliance ──────────────────────────
    workers = group_workers(detections)
    compliance, risk_score, risk_level, violations, breakdown = \
        compute_risk(detections, workers)
    t2 = time.perf_counter()

    # ── Annotate ──────────────────────────────────────────────
    annotated = annotate_frame(frame, detections, workers, compliance, risk_level)
    t3 = time.perf_counter()

    # MUST be uint8 for Gradio
    annotated = annotated.astype(np.uint8)

    detect_ms   = round((t1-t0)*1000, 2)
    validate_ms = round((t2-t1)*1000, 2)
    render_ms   = round((t3-t2)*1000, 2)
    total_ms    = round((t3-t0)*1000, 2)
    fps         = round(1000/total_ms, 1) if total_ms > 0 else 0.0

    compliant_n = sum(1 for w in workers if w.compliance_state=='COMPLIANT')
    semi_n      = sum(1 for w in workers if w.compliance_state=='SEMI-COMPLIANT')
    non_c_n     = sum(1 for w in workers if w.compliance_state=='NON-COMPLIANT')

    if not violations:
        summary = "All workers compliant. No PPE violations detected."
    else:
        nc   = [f"W{w.worker_id}" for w in workers if w.compliance_state=='NON-COMPLIANT']
        semi = [f"W{w.worker_id}" for w in workers if w.compliance_state=='SEMI-COMPLIANT']
        parts = []
        if nc:   parts.append(f"{len(nc)} non-compliant ({', '.join(nc)})")
        if semi: parts.append(f"{len(semi)} semi-compliant ({', '.join(semi)})")
        ctx = " Hazardous machinery detected — severity elevated." \
              if breakdown.get('hazards',0) else ""
        summary = (f"{', '.join(parts)} out of {len(workers)} workers. "
                   f"Risk score: {risk_score:.3f}.{ctx} "
                   f"Corrective action recommended.")

    return InferenceResult(
        annotated_image=annotated,
        detections=detections,
        workers=workers,
        violations=violations,
        compliance_score=compliance,
        risk_level=risk_level,
        risk_score=risk_score,
        person_count=len(workers),
        compliant_count=compliant_n,
        semi_compliant_count=semi_n,
        non_compliant_count=non_c_n,
        summary=summary,
        inference_time_ms=total_ms,
        risk_breakdown=breakdown,
        timing={'detect_ms':detect_ms,'validate_ms':validate_ms,
                'render_ms':render_ms,'total_ms':total_ms,'fps':fps},
        frame_id=frame_id,
    )

# ── Quick sanity check (no model call, just shape test) ───────
_dummy = np.random.randint(60,200,(480,640,3),dtype=np.uint8)
_ann   = annotate_frame(_dummy, [], [], 100.0, 'LOW')
assert _ann.shape == _dummy.shape and _ann.dtype == np.uint8
print("✔  annotate_frame shape/dtype check passed.")
print(f"   Output: {_ann.shape}  dtype:{_ann.dtype}")
print("\n══════════════════════════════════════")
print("  CELL 2 COMPLETE — real model wired")
print("══════════════════════════════════════")

✔  annotate_frame shape/dtype check passed.
   Output: (480, 640, 3)  dtype:uint8

══════════════════════════════════════
  CELL 2 COMPLETE — real model wired
══════════════════════════════════════


In [7]:
import gradio as gr
import numpy as np
import datetime
import cv2
import tempfile
import os
import gc
import json
from tqdm.notebook import tqdm

# ── HTML helpers ──────────────────────────────────────────────
def _badge(status: str) -> str:
    c = {"COMPLIANT":"compliant","SEMI-COMPLIANT":"semi",
         "NON-COMPLIANT":"violation"}.get(status,"semi")
    return f'<span class="badge badge-{c}">{status}</span>'

def _card(value, label, sub, variant="") -> str:
    return (f'<div class="metric-card {variant}">'
            f'<div class="metric-card-value">{value}</div>'
            f'<div class="metric-card-label">{label}</div>'
            f'<div class="metric-card-sub">{sub}</div></div>')

def build_metric_html(r: InferenceResult) -> str:
    return (f'<div class="metric-grid">'
            + _card(r.person_count,         "TOTAL WORKERS",  f"RISK: {r.risk_level}","")
            + _card(r.compliant_count,      "COMPLIANT",       "FULL PPE ✔",  "compliant")
            + _card(r.semi_compliant_count, "SEMI-COMPLIANT",  "PARTIAL PPE", "semi")
            + _card(r.non_compliant_count,  "NON-COMPLIANT",   "PPE MISSING", "violation")
            + '</div>')

def build_metric_html_video(stats: dict) -> str:
    risk = stats.get("worst_risk_level","LOW")
    return (f'<div class="metric-grid">'
            + _card(stats.get("frames_processed",0),
                    "FRAMES ANALYSED",  f"RISK: {risk}",          "")
            + _card(f'{stats.get("avg_compliance",0):.1f}%',
                    "AVG COMPLIANCE",   "ACROSS ALL FRAMES",       "compliant")
            + _card(stats.get("worst_risk_level","—"),
                    "PEAK RISK",        "SCENE MAXIMUM",           "violation")
            + _card(len(stats.get("violation_freq",{})),
                    "VIOLATION TYPES",  "UNIQUE CLASSES FLAGGED",  "semi")
            + '</div>')

def build_telem_strip_image(r: InferenceResult) -> str:
    t = r.timing
    return (f'<div class="telem-strip">'
            f'<span>DETECT   <span class="live">{t["detect_ms"]} ms</span></span>'
            f'<span>VALIDATE <span class="live">{t["validate_ms"]} ms</span></span>'
            f'<span>RENDER   <span class="live">{t["render_ms"]} ms</span></span>'
            f'<span>TOTAL    <span class="live">{t["total_ms"]} ms</span></span>'
            f'<span>FPS      <span class="live">{t["fps"]}</span></span>'
            f'<span>COMPLIANCE <span class="live">{r.compliance_score:.1f}%</span></span>'
            f'<span>RISK     <span class="live">{r.risk_level}</span></span>'
            f'</div>')

def build_telem_strip_video(stats: dict) -> str:
    vfreq    = stats.get("violation_freq", {})
    top_viol = (max(vfreq, key=vfreq.get) if vfreq else "—")
    return (f'<div class="telem-strip">'
            f'<span>FRAMES PROCESSED <span class="live">'
            f'{stats.get("frames_processed",0)}</span></span>'
            f'<span>AVG COMPLIANCE <span class="live">'
            f'{stats.get("avg_compliance",0):.1f}%</span></span>'
            f'<span>PEAK RISK <span class="live">'
            f'{stats.get("worst_risk_level","—")}</span></span>'
            f'<span>TOP VIOLATION <span class="live">{top_viol}</span></span>'
            f'</div>')

def build_worker_table(workers: list) -> str:
    if not workers:
        return (f'<p style="color:#475569;font-family:monospace;padding:12px">'
                f'NO WORKERS DETECTED</p>')
    rows = ""
    for w in workers:
        viols  = ", ".join(w.violations) if w.violations else "—"
        ppe    = ", ".join(w.ppe_worn)   if w.ppe_worn   else "—"
        rl_col = ("#EF4444" if w.risk_level in ("CRITICAL","HIGH")
                  else "#F59E0B" if w.risk_level=="MEDIUM" else "#10B981")
        rows += (f'<tr>'
                 f'<td class="id-col">W{w.worker_id}</td>'
                 f'<td>{_badge(w.compliance_state)}</td>'
                 f'<td style="color:{PALETTE["compliant"]};font-size:11px">{ppe}</td>'
                 f'<td style="color:{PALETTE["violation"]};font-size:11px">{viols}</td>'
                 f'<td>{w.risk_score:.3f}</td>'
                 f'<td style="font-weight:700;color:{rl_col}">{w.risk_level}</td>'
                 f'<td>{datetime.datetime.now().strftime("%H:%M:%S")}</td>'
                 f'</tr>')
    return (f'<table class="telem-grid"><thead><tr>'
            f'<th>WORKER</th><th>STATUS</th><th>PPE WORN</th>'
            f'<th>VIOLATIONS</th><th>RISK SCORE</th><th>RISK LEVEL</th><th>TIME</th>'
            f'</tr></thead><tbody>{rows}</tbody></table>')

def build_violation_html(workers: list, result: InferenceResult) -> str:
    viols = [w for w in workers if w.compliance_state != 'COMPLIANT']
    summary_bar = (
        f'<div style="font-family:{FONT_MONO};font-size:11px;padding:8px 12px;'
        f'margin-bottom:10px;background:{PALETTE["bg_panel_alt"]};'
        f'border-left:3px solid {PALETTE["accent_blue"]}>'
        f'{result.summary}</div>'
    )
    if not viols:
        return (summary_bar +
                f'<div style="padding:16px;text-align:center;'
                f'font-family:{FONT_MONO};color:{PALETTE["compliant"]};'
                f'font-size:13px">✔ &nbsp; ALL WORKERS COMPLIANT</div>')
    rows = summary_bar
    for w in viols:
        col  = (PALETTE["semi"] if w.compliance_state=="SEMI-COMPLIANT"
                else PALETTE["violation"])
        miss = ", ".join(w.violations) if w.violations else "Unknown"
        rows += (f'<div style="border-left:3px solid {col};padding:10px 14px;'
                 f'margin-bottom:8px;background:{PALETTE["bg_panel_alt"]};'
                 f'font-family:{FONT_MONO};font-size:12px">'
                 f'<span style="color:{col};font-weight:700">[{w.compliance_state}]</span>'
                 f' &nbsp; Worker '
                 f'<span style="color:{PALETTE["accent_blue"]}">W{w.worker_id}</span>'
                 f' &nbsp;|&nbsp; Violations: '
                 f'<span style="color:{PALETTE["violation"]}>{miss}</span>'
                 f' &nbsp;|&nbsp; Risk: {w.risk_score:.3f} ({w.risk_level})'
                 f'</div>')
    return f'<div style="padding:8px">{rows}</div>'

def build_video_summary_html(stats: dict) -> str:
    risk_col = {
        "CRITICAL": PALETTE["violation"], "HIGH": "#F97316",
        "MEDIUM":   PALETTE["semi"],      "LOW":  PALETTE["compliant"],
    }.get(stats.get("worst_risk_level","LOW"), PALETTE["compliant"])

    vfreq = stats.get("violation_freq", {})
    freq_rows = ""
    for vtype, count in sorted(vfreq.items(), key=lambda x: -x[1]):
        total_f = max(stats.get("frames_processed",1), 1)
        pct     = round(count / total_f * 100, 1)
        bar_w   = max(2, int(pct))
        freq_rows += (
            f'<tr>'
            f'<td style="color:{PALETTE["violation"]};font-size:11px">{vtype}</td>'
            f'<td style="font-family:{FONT_MONO}">{count}</td>'
            f'<td style="width:40%;padding-right:12px">'
            f'<div style="background:{PALETTE["violation"]};height:8px;'
            f'width:{bar_w}%;border-radius:1px;min-width:4px"></div></td>'
            f'<td style="color:{PALETTE["text_secondary"]};'
            f'font-family:{FONT_MONO}">{pct}%</td>'
            f'</tr>'
        )

    freq_table = (
        f'<table class="telem-grid">'
        f'<thead><tr>'
        f'<th>VIOLATION TYPE</th><th>OCCURRENCES</th>'
        f'<th>FREQUENCY</th><th>% OF FRAMES</th>'
        f'</tr></thead><tbody>'
        + (freq_rows if freq_rows else
           f'<tr><td colspan="4" style="padding:12px;color:{PALETTE["compliant"]};'
           f'font-family:{FONT_MONO}">✔ NO VIOLATIONS DETECTED</td></tr>')
        + '</tbody></table>'
    )

    return f"""
<div style="padding:8px">
  <div style="display:grid;grid-template-columns:repeat(3,1fr);
              gap:10px;margin-bottom:16px">
    <div style="background:{PALETTE['bg_panel']};border:1px solid {PALETTE['border']};
                border-top:2px solid {PALETTE['accent_blue']};padding:14px">
      <div style="font-family:{FONT_MONO};font-size:26px;font-weight:700;
                  color:{PALETTE['text_primary']}">{stats.get('frames_processed',0)}</div>
      <div style="font-size:10px;color:{PALETTE['text_secondary']};
                  text-transform:uppercase;letter-spacing:.08em">FRAMES ANALYSED</div>
      <div style="font-family:{FONT_MONO};font-size:10px;
                  color:{PALETTE['accent_cyan']};margin-top:5px">
        AVG {stats.get('avg_compliance',0):.1f}% COMPLIANCE</div>
    </div>
    <div style="background:{PALETTE['bg_panel']};border:1px solid {PALETTE['border']};
                border-top:2px solid {risk_col};padding:14px">
      <div style="font-family:{FONT_MONO};font-size:26px;font-weight:700;
                  color:{risk_col}">{stats.get('worst_risk_level','—')}</div>
      <div style="font-size:10px;color:{PALETTE['text_secondary']};
                  text-transform:uppercase;letter-spacing:.08em">PEAK RISK LEVEL</div>
      <div style="font-family:{FONT_MONO};font-size:10px;
                  color:{PALETTE['accent_cyan']};margin-top:5px">
        ACROSS ENTIRE VIDEO</div>
    </div>
    <div style="background:{PALETTE['bg_panel']};border:1px solid {PALETTE['border']};
                border-top:2px solid {PALETTE['semi']};padding:14px">
      <div style="font-family:{FONT_MONO};font-size:26px;font-weight:700;
                  color:{PALETTE['semi']}'>{len(vfreq)}</div>
      <div style="font-size:10px;color:{PALETTE['text_secondary']};
                  text-transform:uppercase;letter-spacing:.08em">VIOLATION TYPES</div>
      <div style="font-family:{FONT_MONO};font-size:10px;
                  color:{PALETTE['accent_cyan']};margin-top:5px">
        UNIQUE CLASSES FLAGGED</div>
    </div>
  </div>
  <div class="ppe-panel-title">▸ VIOLATION FREQUENCY BREAKDOWN</div>
  {freq_table}
</div>"""


# ── VIDEO CORE ────────────────────────────────────────────────
def process_video(video_path: str,
                  conf_threshold: float = 0.45,
                  sample_every_n: int = 2,
                  max_frames: int = 500) -> dict:
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise IOError(f"Cannot open video: {video_path}")

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps          = cap.get(cv2.CAP_PROP_FPS) or 25.0
    orig_w       = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    orig_h       = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    out_path = tempfile.mktemp(suffix=".mp4")
    fourcc   = cv2.VideoWriter_fourcc(*"mp4v")
    out_fps  = max(1.0, fps / sample_every_n)
    writer   = cv2.VideoWriter(out_path, fourcc, out_fps, (orig_w, orig_h))

    frame_results  = []
    frame_idx      = 0
    processed      = 0
    all_violations = []

    print(f"Video: {total_frames} frames @ {fps:.1f} fps  |  "
          f"Sampling every {sample_every_n} frame(s)  |  "
          f"Cap: {max_frames} frames")

    pbar = tqdm(
        total=min(total_frames // max(sample_every_n,1), max_frames),
        desc="Analysing", unit="frame"
    )

    while cap.isOpened() and processed < max_frames:
        ret, bgr_frame = cap.read()
        if not ret:
            break
        frame_idx += 1
        if frame_idx % sample_every_n != 0:
            continue

        rgb_frame = cv2.cvtColor(bgr_frame, cv2.COLOR_BGR2RGB)
        result    = run_inference(rgb_frame,
                                  conf_threshold=conf_threshold,
                                  frame_id=frame_idx)
        writer.write(cv2.cvtColor(result.annotated_image, cv2.COLOR_RGB2BGR))

        frame_results.append({
            'frame_id':        frame_idx,
            'compliance':      result.compliance_score,
            'risk_level':      result.risk_level,
            'violation_count': len(result.violations),
            'person_count':    result.person_count,
        })
        all_violations.extend([v['label'] for v in result.violations])
        processed += 1
        pbar.update(1)

    pbar.close()
    cap.release()
    writer.release()
    gc.collect()

    RISK_ORDER = ['LOW','MEDIUM','HIGH','CRITICAL']
    if frame_results:
        avg_compliance = round(
            np.mean([f['compliance'] for f in frame_results]), 1)
        worst_risk = max(
            frame_results,
            key=lambda f: RISK_ORDER.index(f['risk_level'])
        )['risk_level']
    else:
        avg_compliance = 100.0
        worst_risk     = 'LOW'

    violation_freq = {}
    for v in all_violations:
        violation_freq[v] = violation_freq.get(v, 0) + 1

    print(f"\n✔  Done — {processed} frames processed.")
    print(f"   Avg Compliance : {avg_compliance}%")
    print(f"   Worst Risk     : {worst_risk}")
    print(f"   Violations     : {violation_freq}")

    return {
        'output_path':      out_path,
        'frames_processed': processed,
        'avg_compliance':   avg_compliance,
        'worst_risk_level': worst_risk,
        'violation_freq':   violation_freq,
        'frame_results':    frame_results,
    }


def _slider_to_step(target_fps: int, source_fps: float) -> int:
    return max(1, int(round(source_fps / target_fps)))


# ── IMAGE CALLBACKS ───────────────────────────────────────────
def run_scan(image: np.ndarray):
    if image is None:
        return (None, EMPTY_M, EMPTY_T,
                build_worker_table([]),
                gr.update(visible=False),
                gr.update(value="⬡  SCAN", interactive=True))
    result = run_inference(image)
    return (
        result.annotated_image,
        build_metric_html(result),
        build_telem_strip_image(result),
        build_worker_table(result.workers),
        gr.update(visible=True),
        gr.update(value="⬡  RE-SCAN", interactive=True),
    )

def run_clear_image():
    return (None, None, EMPTY_M, EMPTY_T,
            build_worker_table([]),
            gr.update(visible=False),
            gr.update(visible=False),
            gr.update(value="⬡  SCAN", interactive=True))

def run_export_image(image: np.ndarray):
    if image is None:
        return gr.update(visible=False)
    result = run_inference(image)
    pdf  = generate_compliance_pdf(
        workers=result.workers,
        detections=result.detections,
        compliance_score=result.compliance_score,
        risk_level=result.risk_level,
    )
    ts   = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    path = f"/tmp/PPE_ImageScan_{ts}.pdf"
    with open(path,"wb") as f: f.write(pdf)
    return gr.update(value=path, visible=True)


# ── VIDEO CALLBACKS ───────────────────────────────────────────
def run_video_scan(video_path, target_fps: int):
    """
    Runs video processing, stores stats in gr.State for PDF export.
    Returns stats as JSON string so export never needs to re-process.
    """
    if video_path is None:
        return (None, EMPTY_M, EMPTY_T, EMPTY_D,
                gr.update(visible=False),
                gr.update(value="⬡  ANALYSE VIDEO", interactive=True),
                "{}")          # empty state
    try:
        cap     = cv2.VideoCapture(video_path)
        src_fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
        cap.release()
        step  = _slider_to_step(int(target_fps), src_fps)
        stats = process_video(video_path,
                              conf_threshold=0.45,
                              sample_every_n=step,
                              max_frames=500)
    except Exception as e:
        err = (f'<p style="color:{PALETTE["violation"]};font-family:{FONT_MONO};'
               f'padding:16px">ERROR: {str(e)}</p>')
        return (None, EMPTY_M, EMPTY_T, err,
                gr.update(visible=False),
                gr.update(value="⬡  ANALYSE VIDEO", interactive=True),
                "{}")

    # Serialise stats to JSON for gr.State
    # frame_results kept but violation_freq is what PDF needs
    stats_json = json.dumps({
        "frames_processed": stats["frames_processed"],
        "avg_compliance":   stats["avg_compliance"],
        "worst_risk_level": stats["worst_risk_level"],
        "violation_freq":   stats["violation_freq"],
    })

    return (
        stats["output_path"],
        build_metric_html_video(stats),
        build_telem_strip_video(stats),
        build_video_summary_html(stats),
        gr.update(visible=True),
        gr.update(value="⬡  RE-ANALYSE", interactive=True),
        stats_json,            # → stored in vid_stats_state
    )


def run_export_video(stats_json: str):
    """
    Builds PDF entirely from stored stats — NO re-processing.
    Uses violation_freq, avg_compliance, worst_risk_level from state.
    """
    if not stats_json or stats_json == "{}":
        return gr.update(visible=False)

    try:
        stats = json.loads(stats_json)
    except Exception:
        return gr.update(visible=False)

    avg_compliance  = stats.get("avg_compliance",  0.0)
    worst_risk      = stats.get("worst_risk_level","LOW")
    frames_done     = stats.get("frames_processed", 0)
    violation_freq  = stats.get("violation_freq",  {})

    # Build Worker-like summary rows from violation_freq for the PDF table
    # We create pseudo-workers: one row per violation type detected
    # so the PDF table is populated with real data
    from dataclasses import dataclass, field as dc_field

    @dataclass
    class _PseudoWorker:
        worker_id:        int
        bbox:             list
        confidence:       float
        ppe_worn:         list
        violations:       list
        compliance_state: str
        risk_score:       float
        risk_level:       str

    pseudo_workers = []
    RISK_ORDER = ['LOW','MEDIUM','HIGH','CRITICAL']

    for idx, (vtype, count) in enumerate(
            sorted(violation_freq.items(), key=lambda x: -x[1]), start=1):
        # Risk level derived from violation weight
        w = VIOLATION_WEIGHTS.get(vtype, 0.5)
        rs = round(w * (count / max(frames_done, 1)), 4)
        rl = next((lvl for lvl, th in RISK_THRESHOLDS if rs >= th), 'LOW')
        pseudo_workers.append(_PseudoWorker(
            worker_id=idx, bbox=[0,0,0,0], confidence=0.0,
            ppe_worn=[], violations=[vtype] * count,
            compliance_state="NON-COMPLIANT",
            risk_score=rs, risk_level=rl,
        ))

    pdf = generate_compliance_pdf(
        workers=pseudo_workers,
        detections=[],
        compliance_score=avg_compliance,
        risk_level=worst_risk,
        site_name="INDUSTRIAL COMPLEX — ZONE 7  [VIDEO SCAN]",
        inspector=f"AUTOMATED VISION SYSTEM  ·  {frames_done} FRAMES ANALYSED",
    )
    ts   = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    path = f"/tmp/PPE_VideoScan_{ts}.pdf"
    with open(path,"wb") as f: f.write(pdf)
    return gr.update(value=path, visible=True)


def run_clear_video():
    return (None, None, EMPTY_M, EMPTY_T, EMPTY_D,
            gr.update(visible=False),
            gr.update(visible=False),
            gr.update(value="⬡  ANALYSE VIDEO", interactive=True),
            "{}")


# ── STATIC HTML ───────────────────────────────────────────────
HEADER_HTML = f"""
<div class="ppe-header">
  <div>
    <div class="ppe-header-title">⬡ &nbsp;PPE COMPLIANCE MONITOR</div>
    <div class="ppe-header-subtitle">
      Industrial Safety Intelligence System
      &nbsp;·&nbsp; YOLOv11 Vision Engine
      &nbsp;·&nbsp; Zone Control Dashboard
    </div>
  </div>
  <div class="sys-status-online">SYSTEM ONLINE</div>
</div>"""

EMPTY_M = (f'<div class="metric-grid">'
           + _card(0,"TOTAL WORKERS","AWAITING SCAN","")
           + _card(0,"COMPLIANT","FULL PPE ✔","compliant")
           + _card(0,"SEMI-COMPLIANT","PARTIAL PPE","semi")
           + _card(0,"NON-COMPLIANT","PPE MISSING","violation")
           + '</div>')
EMPTY_T = '<div class="telem-strip"><span>AWAITING SCAN…</span></div>'
EMPTY_D = (f'<p style="color:{PALETTE["text_secondary"]};font-family:{FONT_MONO};'
           f'padding:16px;font-size:11px">UPLOAD CONTENT AND CLICK SCAN</p>')

# ── SLIDER INPUT BOX CSS FIX ──────────────────────────────────
# Appended to INDUSTRIAL_CSS already defined in Cell 1.
# Targets the number input box that appears alongside gr.Slider.
SLIDER_FIX_CSS = """
/* ── Slider number input box — force visible text ── */
input[type='number'] {
    color: #F1F5F9 !important;
    background: #1E293B !important;
    border: 1px solid #475569 !important;
    border-radius: 2px !important;
    font-family: 'JetBrains Mono', monospace !important;
    font-size: 12px !important;
    font-weight: 600 !important;
    padding: 2px 6px !important;
}
input[type='number']:focus {
    border-color: #06B6D4 !important;
    outline: none !important;
    box-shadow: 0 0 0 2px rgba(6,182,212,0.2) !important;
}
/* Slider track */
input[type='range'] {
    accent-color: #3B82F6 !important;
}
"""

# ── LAYOUT ────────────────────────────────────────────────────
custom_theme = gr.themes.Base(
    primary_hue="blue", neutral_hue="slate",
    font=gr.themes.GoogleFont("Inter"),
).set(
    body_background_fill="#0F172A",
    block_background_fill="#1E293B",
    block_border_color="#334155",
    block_border_width="1px",
    block_radius="2px",
    input_background_fill="#162032",
    input_border_color="#334155",
    button_primary_background_fill="#3B82F6",
    button_primary_border_color="#3B82F6",
)

with gr.Blocks(
    css=INDUSTRIAL_CSS + SLIDER_FIX_CSS,   # ← append fix
    title="PPE Compliance Monitor",
    theme=custom_theme,
) as demo:

    gr.HTML(HEADER_HTML)
    metric_display = gr.HTML(value=EMPTY_M)
    telem_display  = gr.HTML(value=EMPTY_T)

    # ── State: stores last video scan stats for PDF export ────
    vid_stats_state = gr.State(value="{}")

    with gr.Tabs():

        # ══════════════════════════════════════════════════════
        # TAB 1 — IMAGE SCAN
        # ══════════════════════════════════════════════════════
        with gr.Tab("⬡  IMAGE SCAN"):
            with gr.Row(equal_height=True):

                with gr.Column(scale=1, min_width=280):
                    gr.HTML('<div class="ppe-panel-title">▸ INPUT FRAME</div>')
                    image_input = gr.Image(
                        label="Upload Image / Photo",
                        type="numpy",
                        height=420,
                    )
                    with gr.Row():
                        img_scan_btn  = gr.Button("⬡  SCAN",
                                                   variant="primary",  size="lg")
                        img_clear_btn = gr.Button("CLEAR",
                                                   variant="secondary", size="lg")
                    img_export_btn = gr.Button(
                        "⬇  EXPORT PDF REPORT",
                        variant="secondary", visible=False)
                    img_pdf_out = gr.File(
                        label="Download Report",
                        visible=False, interactive=False)

                with gr.Column(scale=2):
                    gr.HTML('<div class="ppe-panel-title">▸ ANNOTATED OUTPUT</div>')
                    image_output = gr.Image(
                        label="",
                        type="numpy",
                        interactive=False,
                        height=420,
                        show_download_button=True,
                    )
                    gr.HTML(
                        f'<div style="font-family:{FONT_MONO};font-size:10px;'
                        f'color:{PALETTE["text_secondary"]};padding:6px 2px;'
                        f'letter-spacing:.06em">'
                        f'■ <span style="color:#1A9E4A">COMPLIANT</span>'
                        f'&nbsp;&nbsp;'
                        f'■ <span style="color:#D4A017">SEMI-COMPLIANT</span>'
                        f'&nbsp;&nbsp;'
                        f'■ <span style="color:#D92B2B">NON-COMPLIANT</span>'
                        f'&nbsp;&nbsp;·&nbsp;&nbsp;'
                        f'Corner ticks + W-label per detected worker'
                        f'</div>'
                    )

            gr.HTML('<div class="ppe-panel-title" style="margin-top:14px">'
                    '▸ WORKER DETECTION TELEMETRY</div>')
            img_worker_table = gr.HTML(value=build_worker_table([]))

        # ══════════════════════════════════════════════════════
        # TAB 2 — VIDEO ANALYSIS
        # ══════════════════════════════════════════════════════
        with gr.Tab("▶  VIDEO ANALYSIS"):
            with gr.Row(equal_height=True):

                with gr.Column(scale=1, min_width=280):
                    gr.HTML('<div class="ppe-panel-title">▸ VIDEO INPUT</div>')
                    video_input = gr.Video(
                        label="Upload Video  (MP4 / AVI / MOV)",
                        height=320,
                    )
                    gr.HTML(
                        f'<div style="font-family:{FONT_MONO};font-size:10px;'
                        f'color:{PALETTE["accent_cyan"]};letter-spacing:.08em;'
                        f'text-transform:uppercase;margin:16px 0 4px 0">'
                        f'▸ ANALYSIS FRAME RATE (FPS)</div>'
                        f'<div style="font-family:{FONT_SANS};font-size:10px;'
                        f'color:{PALETTE["text_secondary"]};margin-bottom:8px">'
                        f'Lower = faster · Higher = more detail · '
                        f'Recommended: 5 FPS</div>'
                    )
                    fps_slider = gr.Slider(
                        minimum=1, maximum=10,
                        value=5, step=1,
                        label="Frames Per Second to Analyse",
                        container=True,
                    )
                    gr.HTML(
                        f'<div style="font-family:{FONT_SANS};font-size:10px;'
                        f'color:{PALETTE["text_secondary"]};'
                        f'padding:6px 0 14px 0;line-height:1.6">'
                        f'1 FPS → fastest, skips most frames<br>'
                        f'5 FPS → balanced (default)<br>'
                        f'10 FPS → most detail, slowest<br>'
                        f'⚠ Max 500 frames · GPU recommended &gt;30s clips'
                        f'</div>'
                    )
                    with gr.Row():
                        vid_scan_btn  = gr.Button("⬡  ANALYSE VIDEO",
                                                   variant="primary",  size="lg")
                        vid_clear_btn = gr.Button("CLEAR",
                                                   variant="secondary", size="lg")
                    vid_export_btn = gr.Button(
                        "⬇  EXPORT VIDEO REPORT",
                        variant="secondary", visible=False)
                    vid_pdf_out = gr.File(
                        label="Download Report",
                        visible=False, interactive=False)

                with gr.Column(scale=2):
                    gr.HTML('<div class="ppe-panel-title">▸ ANNOTATED VIDEO OUTPUT</div>')
                    video_output = gr.Video(
                        label="",
                        height=320,
                        interactive=False,
                        show_download_button=True,
                    )
                    gr.HTML(
                        f'<div style="font-family:{FONT_MONO};font-size:10px;'
                        f'color:{PALETTE["text_secondary"]};padding:6px 2px;'
                        f'letter-spacing:.06em">'
                        f'■ <span style="color:#1A9E4A">COMPLIANT</span>'
                        f'&nbsp;&nbsp;'
                        f'■ <span style="color:#D4A017">SEMI-COMPLIANT</span>'
                        f'&nbsp;&nbsp;'
                        f'■ <span style="color:#D92B2B">NON-COMPLIANT</span>'
                        f'&nbsp;&nbsp;·&nbsp;&nbsp;'
                        f'Full PPE + worker annotations on every sampled frame'
                        f'</div>'
                    )

            gr.HTML('<div class="ppe-panel-title" style="margin-top:14px">'
                    '▸ VIDEO ANALYSIS SUMMARY</div>')
            vid_summary = gr.HTML(value=EMPTY_D)


    # ── EVENT WIRING ──────────────────────────────────────────

    # Image tab
    img_scan_btn.click(
        fn=run_scan,
        inputs=[image_input],
        outputs=[image_output, metric_display, telem_display,
                 img_worker_table, img_export_btn, img_scan_btn],
    )
    img_clear_btn.click(
        fn=run_clear_image, inputs=[],
        outputs=[image_input, image_output, metric_display, telem_display,
                 img_worker_table, img_export_btn, img_pdf_out, img_scan_btn],
    )
    img_export_btn.click(
        fn=run_export_image,
        inputs=[image_input],
        outputs=[img_pdf_out],
    )

    # Video tab — note vid_stats_state in outputs of scan, input of export
    vid_scan_btn.click(
        fn=run_video_scan,
        inputs=[video_input, fps_slider],
        outputs=[video_output, metric_display, telem_display,
                 vid_summary, vid_export_btn, vid_scan_btn,
                 vid_stats_state],          # ← stores stats here
    )
    vid_clear_btn.click(
        fn=run_clear_video, inputs=[],
        outputs=[video_input, video_output, metric_display, telem_display,
                 vid_summary, vid_export_btn, vid_pdf_out,
                 vid_scan_btn, vid_stats_state],
    )
    vid_export_btn.click(
        fn=run_export_video,
        inputs=[vid_stats_state],           # ← reads from state, no re-process
        outputs=[vid_pdf_out],
    )


# ── LAUNCH ────────────────────────────────────────────────────
print("Launching PPE Compliance Monitor…")
demo.launch(
    server_name="0.0.0.0",
    server_port=0,
    share=True,
    show_error=True
)

/tmp/ipykernel_40208/3047270308.py:523: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_40208/3047270308.py:523: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(


Launching PPE Compliance Monitor…
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1a6c90dbb31b8215fd.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
